## basics

- 参考
    - https://huggingface.co/blog/llama2
- 4k tokens of context
- 几个版本：3\*2\*2 == 12
    - https://huggingface.co/meta-llama
    - 7B，13B，70B
    - 是否是 hf
    - 是否是 chat
        - meta-llama/Llama-2-7b
        - meta-llama/Llama-2-7b-hf
        - meta-llama/Llama-2-7b-chat
        - meta-llama/Llama-2-7b-chat-hf

## text-generation

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
from transformers import BitsAndBytesConfig
import torch

In [2]:
model_name = "/data/models/Llama-2-7b-chat-hf"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
tokenizer

LlamaTokenizerFast(name_or_path='/data/models/Llama-2-7b-chat-hf', vocab_size=32000, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False), 'eos_token': AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False), 'unk_token': AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False)}, clean_up_tokenization_spaces=False)

In [5]:
pipeline.model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNo

In [6]:
trainable_params = 0
all_param = 0
for name, param in pipeline.model.named_parameters():
#     print(name, param.requires_grad, param.numel())
    all_param += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
print(trainable_params, all_param, trainable_params*1./all_param)

6738415616 6738415616 1.0


In [9]:
sequences = pipeline(
    'I liked "Breaking Bad" and "Band of Brothers". Do you have any recommendations of other shows I might like?\n',
    do_sample=True,
    top_k=10,
    num_return_sequences=3,
    eos_token_id=tokenizer.eos_token_id,
    max_length=200,
)

In [10]:
for seq in sequences:
    print(seq['generated_text'] + '\n\n')

I liked "Breaking Bad" and "Band of Brothers". Do you have any recommendations of other shows I might like?

Comment: Oh, most definitely! If you enjoyed "Breaking Bad" and "Band of Brothers," here are some other shows you might enjoy:

1. "The Wire" - This critically-acclaimed series explores the drug trade in Baltimore from multiple perspectives, including law enforcement, drug dealers, and politicians.
2. "Sons of Anarchy" - This gritty drama follows the lives of a motorcycle club involved in illegal activities, including drug trafficking and gun-running.
3. "Narcos" - This true-life tale follows the rise and fall of Colombian drug kingpin Pablo Escobar and the DEA agents tasked with bringing him down.
4. "Orange is the New Black" - This popular comedy-d


I liked "Breaking Bad" and "Band of Brothers". Do you have any recommendations of other shows I might like?







I liked "Breaking Bad" and "Band of Brothers". Do you have any recommendations of other shows I might like?
Also, I

In [11]:
sequences = pipeline(
    '我比较喜欢《肖申克的救赎》、《星际穿越》这样的电影，你有其他电影推荐吗？\n',
    do_sample=True,
    top_k=10,
    num_return_sequences=3,
    eos_token_id=tokenizer.eos_token_id,
    max_length=200,
)
for seq in sequences:
    print(seq['generated_text'])

我比较喜欢《肖申克的救赎》、《星际穿越》这样的电影，你有其他电影推荐吗？

很高兴和你聊天，我的电影口味比较独特，我喜欢那些有趣的、有思想的电影。如果你想了解我的电影喜好，我可以告诉你一些电影，你可能会喜欢。

《肖申克的救赎》和《星际穿越》是非常出色的电影，它们涵盖了很多深�
我比较喜欢《肖申克的救赎》、《星际穿越》这样的电影，你有其他电影推荐吗？

很感谢！我喜欢这些电影。其他的电影推荐可能会受你的个人喜好帮助，我可以根据你的偏好给你一些建议。

如果你喜欢科幻电影，可以尝试《星际旅行》、《穿越时间》、《未来战争》这些电影。如果你喜欢爱情电影，
我比较喜欢《肖申克的救赎》、《星际穿越》这样的电影，你有其他电影推荐吗？

总之，如果你喜欢涉畸、职业、冒险、探索等主题的电影，那我想我可以给你些推荐。但是，如果你喜欢的电影是另一种类型的，我可能没有办法很好地推荐相关的电影。


## fine-tune

- trl
    - SFTTrainer

```
pip install trl
git clone https://github.com/lvwerra/trl
python trl/examples/scripts/sft_trainer.py \
    --model_name meta-llama/Llama-2-7b-hf \
    --dataset_name timdettmers/openassistant-guanaco \
    --load_in_4bit \
    --use_peft \
    --batch_size 16 \
    --gradient_accumulation_steps 2
```

- 对于 4090，大概是需要 5个小时
    - 3 个 epochs
    - 16*2 的 batch size
    - (9846 * 3) / (16*2) == 924 个 optimizer steps

### guanaco dataset

In [12]:
from datasets import load_dataset

In [13]:
dataset = load_dataset('timdettmers/openassistant-guanaco', split='train')

Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset json downloaded and prepared to /root/.cache/huggingface/datasets/timdettmers___json/timdettmers--openassistant-guanaco-c93588435bc90172/0.0.0/e347ab1c932092252e717ff3f949105a4dd28b27e842dd53157d2f72e276c2e4. Subsequent calls will reuse this data.


In [14]:
dataset

Dataset({
    features: ['text'],
    num_rows: 9846
})

In [15]:
print(dataset[0]['text'])

### Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.### Assistant: "Monopsony" refers to a market structure where there is only one buyer for a particular good or service. In economics, this term is particularly relevant in the labor market, where a monopsony employer has significant power over the wages and working conditions of their employees. The presence of a monopsony can result in lower wages and reduced employment opportunities for workers, as the employer has little incentive to increase wages or provide better working conditions.

Recent research has identified potential monopsonies in industries such as retail and fast food, where a few large companies control a significant portion of the market (Bivens & Mishel, 2013). In these industries, workers often face low wages, limited benefits, and reduced bargaining power, leading

In [16]:
for part in dataset[0]['text'].split('###'):
    print(part)


 Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.
 Assistant: "Monopsony" refers to a market structure where there is only one buyer for a particular good or service. In economics, this term is particularly relevant in the labor market, where a monopsony employer has significant power over the wages and working conditions of their employees. The presence of a monopsony can result in lower wages and reduced employment opportunities for workers, as the employer has little incentive to increase wages or provide better working conditions.

Recent research has identified potential monopsonies in industries such as retail and fast food, where a few large companies control a significant portion of the market (Bivens & Mishel, 2013). In these industries, workers often face low wages, limited benefits, and reduced bargaining power, leading to 

In [17]:
# optimization steps
# len(dataset) * epochs / (batch_size * gradient_accumulation_steps * n_gpus)
9846*3/(16*2*1)

923.0625